In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
import random
from scipy import integrate

from benchmark_utils import (
    load_bulk_facs,
    load_cti,
    load_bulk_facs_tpm,
    create_uniform_pseudobulk_dataset,
    create_dirichlet_pseudobulk_dataset,
    preprocess_scrna,
    add_cell_types_grouped,
)

In [ ]:
# --- Load datasets ---
cti_data = load_cti(n_variable_genes=None)
cti_data = preprocess_scrna(cti_data["dataset"], keep_genes=None, batch_key="donor_id")
cti_data, _ = add_cell_types_grouped(cti_data, "2nd_level_granularity")
single_cell_data = cti_data.to_df()

bulk_facs_data = load_bulk_facs()["dataset"].T
bulk_facs_tpm_data = load_bulk_facs_tpm()["dataset"].T

In [ ]:
# --- Common genes across datasets ---
common_genes = list(set(bulk_facs_data.columns) & set(bulk_facs_tpm_data.columns) & set(single_cell_data.columns))

In [4]:
X = cti_data[:, common_genes].X

In [9]:
group_codes, groups = pd.factorize(cti_data.obs["cell_types_grouped_2nd_level_granularity"].values,sort=True) 

In [10]:
to_remove_mask = cti_data.obs["cell_types_grouped_2nd_level_granularity"] == "To remove"

In [11]:
X_clean = X[~to_remove_mask]

In [ ]:
X_clean.shape

In [13]:
sc_data = {}
for code, group in enumerate(groups):
    if group != "To remove":
        sc_data[group] = X[group_codes == code]

### Here we calculate the most variable genes across cell types but lowest within cell type

In [12]:
def calculate_fisher_r2(X_clean, sc_data):
    SS_group = {}
    mu_group = {}
    n_k = {}

    for group in sc_data.keys():
        group_data = sc_data[group].toarray()
        SS_group[group] = np.var(group_data, axis=0)
        n_k[group] = group_data.shape[0]
        SS_group[group] = SS_group[group] * n_k[group]
        mu_group[group] = np.mean(group_data, axis=0)

    mu_total = np.mean(X_clean.toarray(), axis=0)
    SS_between = np.zeros(X_clean.shape[1])

    for group in sc_data.keys():
        SS_between += n_k[group] * (mu_group[group] - mu_total)**2
        
    SS_within = np.sum([SS_group[group] for group in sc_data.keys()], axis=0)
    fisher_r2 = SS_between / (SS_between + SS_within)

    return fisher_r2

In [15]:
fisher_r2 = calculate_fisher_r2(X_clean, sc_data)

In [114]:
# Create a pandas Series with gene names as index and fisher_r2 values
fisher_r2_series = pd.Series(fisher_r2, index=common_genes)

In [116]:
fisher_r2_series.sort_values(ascending=False, inplace=True)

In [ ]:
fisher_r2_series.head(2000)

In [122]:
highest_r2_genes = list(fisher_r2_series.index[:2000])

In [124]:
import pickle
with open('project/highest_r2_genes_2nd_gran.pkl', 'wb') as f:
    pickle.dump(highest_r2_genes, f)

### Adjusted Fisher R2

In the previous metric we might favour the low expressed genes so we use the weighted coefficient of variation instead of the SS_within.

This way we lose the meaning of the percentage of variance compared to above.

In [11]:
from tqdm import tqdm

def calculate_adjusted_fisher_r2(X_clean, sc_data):
    
    CV_group = {}
    mu_group = {}
    n_k = {}
    N = X_clean.shape[0]

    for group in tqdm(sc_data.keys()):
        group_data = sc_data[group].toarray()
        CV_group[group] = np.sqrt(np.var(group_data, axis=0))
        mu_group[group] = np.mean(group_data, axis=0)
        n_k[group] = group_data.shape[0]
        CV_group[group] = CV_group[group] / (mu_group[group]+1e-10)
        CV_group[group] = CV_group[group]

    mu_total = np.mean(X_clean.toarray(), axis=0)
    SS_between = np.zeros(X_clean.shape[1])

    for group in tqdm(sc_data.keys()):
        SS_between += n_k[group] * (mu_group[group] - mu_total)**2
        
    CV_pooled = np.sum([(CV_group[group] * n_k[group]/N)**2 for group in sc_data.keys()], axis=0)
    adjusted_fisher_r2 = SS_between / CV_pooled

    return adjusted_fisher_r2

In [ ]:
adjusted_fisher_r2 = calculate_adjusted_fisher_r2(X_clean, sc_data)

In [12]:
adjusted_fisher_r2 = pd.Series(adjusted_fisher_r2, index=common_genes)
adjusted_fisher_r2.sort_values(ascending=False, inplace=True)
highest_adjr2_genes = list(adjusted_fisher_r2.index[:2000])

In [ ]:
highest_adjr2_genes

In [14]:
import pickle
with open('project/highest_adj_r2_genes_2nd_gran.pkl', 'wb') as f:
    pickle.dump(highest_adjr2_genes, f)

## Overlap with signature matrix of 2nd granularity

In [26]:
# Load marker genes
with open('project/marker_genes_2nd_gran_ensembl.pkl', 'rb') as f:
    marker_genes = pickle.load(f)

In [ ]:
marker_genes

In [85]:
with open('project/most_variable_genes_2nd_gran.pkl', 'rb') as f:
    most_variable_genes = pickle.load(f)

with open('project/low_CV_genes.pkl', 'rb') as f:
    low_CV_genes = pickle.load(f)

with open('project/highest_r2_genes_2nd_gran.pkl', 'rb') as f:
    highest_r2_genes = pickle.load(f)

with open('project/highest_adj_r2_genes_2nd_gran.pkl', 'rb') as f:
    highest_adj_r2_genes = pickle.load(f)

In [86]:
# Calculate overlap percentages for each method and cell type
methods = {
    'Most variable genes': most_variable_genes,
    'Low CV genes': low_CV_genes, 
    'Highest R2 genes': highest_r2_genes,
    'Highest adjusted R2 genes': highest_adj_r2_genes
}

overlap_results = {}

for method_name, method_genes in methods.items():
    overlap_results[method_name] = {}
    
    for cell_type, marker_gene_list in marker_genes.items():
        overlap = len(set(marker_gene_list) & set(method_genes))
        overlap_percentage = (overlap / len(marker_gene_list)) * 100
        overlap_results[method_name][cell_type] = overlap_percentage



In [ ]:
# Convert to DataFrame for easier viewing
import pandas as pd
overlap_df = pd.DataFrame(overlap_results)
print("\nOverlap percentages with marker genes by cell type:")
overlap_df

In [ ]:
# Create bar plot comparing methods
import matplotlib.pyplot as plt

# Set figure size
plt.figure(figsize=(12, 6))

# Get positions for bars
x = np.arange(len(overlap_df.index))
width = 0.2  # Width of bars

# Plot bars for each method
plt.bar(x - width*1.5, overlap_df['Most variable genes'], width, label='Most variable genes')
plt.bar(x - width/2, overlap_df['Low CV genes'], width, label='Low CV genes') 
plt.bar(x + width/2, overlap_df['Highest R2 genes'], width, label='Highest R2 genes')
plt.bar(x + width*1.5, overlap_df['Highest adjusted R2 genes'], width, label='Highest adjusted R2 genes')

# Customize plot
plt.xlabel('Cell Types')
plt.ylabel('Overlap with Marker Genes (%)')
plt.title('Comparison of Gene Selection Methods')
plt.xticks(x, overlap_df.index, rotation=45, ha='right')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Adjust layout to prevent label cutoff
plt.tight_layout()

plt.show()


## Overlap analysis on 3rd Granularity

In [ ]:
## Load CTI data for 3rd granularity
cti_data_3rd_gran = load_cti(n_variable_genes=None)
cti_data_3rd_gran = preprocess_scrna(cti_data_3rd_gran["dataset"], keep_genes=None, batch_key="donor_id")
cti_data_3rd_gran, _ = add_cell_types_grouped(cti_data_3rd_gran, "3rd_level_granularity")
single_cell_data_3rd_gran = cti_data_3rd_gran.to_df()

In [ ]:
# --- Common genes across datasets ---
X = cti_data_3rd_gran.X
group_codes, groups = pd.factorize(cti_data_3rd_gran.obs["cell_types_grouped_3rd_level_granularity"].values,sort=True) 
to_remove_mask = cti_data_3rd_gran.obs["cell_types_grouped_3rd_level_granularity"] == "To remove"
X_clean = X[~to_remove_mask]
X_clean.shape

In [ ]:
# --- Common genes across datasets ---
common_genes = list(set(bulk_facs_data.columns) & set(bulk_facs_tpm_data.columns) & set(single_cell_data.columns))
X = cti_data_3rd_gran[:, common_genes].X
group_codes, groups = pd.factorize(cti_data_3rd_gran.obs["cell_types_grouped_3rd_level_granularity"].values,sort=True) 
to_remove_mask = cti_data_3rd_gran.obs["cell_types_grouped_3rd_level_granularity"] == "To remove"
X_clean = X[~to_remove_mask]
X_clean.shape

In [ ]:
sc_data = {}
for code, group in enumerate(groups):
    if group != "To remove":
        sc_data[group] = X[group_codes == code]

### Fisher R2 ordered genes for 3rd granularity

In [45]:
fisher_r2_3rd_gran = calculate_fisher_r2(X_clean, sc_data)
fisher_r2_3rd_gran = pd.Series(fisher_r2_3rd_gran, index=common_genes)
fisher_r2_3rd_gran.sort_values(ascending=False, inplace=True)
highest_r2_genes_3rd_gran = list(fisher_r2_3rd_gran.index[:2000])

In [ ]:
fisher_r2_3rd_gran.head(2000)

In [47]:
with open('project/highest_r2_genes_3rd_gran.pkl', 'wb') as f:
    pickle.dump(highest_r2_genes_3rd_gran, f)

### Adjusted R2 ordered genes for 3rd granularity

In [ ]:
adjusted_fisher_r2_3rd_gran = calculate_adjusted_fisher_r2(X_clean, sc_data)
adjusted_fisher_r2_3rd_gran = pd.Series(adjusted_fisher_r2_3rd_gran, index=common_genes)
adjusted_fisher_r2_3rd_gran.sort_values(ascending=False, inplace=True)
highest_adjr2_genes_3rd_gran = list(adjusted_fisher_r2_3rd_gran.index[:2000])

In [ ]:
adjusted_fisher_r2_3rd_gran.head(2000)

In [50]:
with open('project/highest_adj_r2_genes_3rd_gran.pkl', 'wb') as f:
    pickle.dump(highest_adjr2_genes_3rd_gran, f)

### Compare to the signature matrix of 3rd granularity

In [ ]:
sc_data.keys()

In [63]:
# Saving to use with rowsetta
marker_genes_3rd_gran = {}

for group in sc_data.keys():
    group_df = pd.read_csv(f'../project/Simon/signature_3rd_level_granularity/DE_{group}.txt', sep='\t', index_col=0)
    marker_genes_3rd_gran[group] = list(group_df.index)
    
with open('project/marker_genes_3rd_gran.pkl', 'wb') as f:
    pickle.dump(marker_genes_3rd_gran, f)

In [ ]:
with open('project/marker_genes_3rd_gran_ensembl.pkl', 'rb') as f:
    marker_genes_3rd_gran_ensembl = pickle.load(f)

marker_genes_3rd_gran_ensembl.keys()

In [81]:
methods_3rd_gran = {
    'Most variable genes': most_variable_genes,
    'Low CV genes': low_CV_genes,
    'Highest R2 genes': highest_r2_genes_3rd_gran,
    'Highest adjusted R2 genes': highest_adjr2_genes_3rd_gran
}

overlap_results_3rd_gran = {}

for method_name, method_genes in methods_3rd_gran.items():
    overlap_results_3rd_gran[method_name] = {}
    
    for cell_type, marker_gene_list in marker_genes_3rd_gran_ensembl.items():
        overlap = len(set(marker_gene_list) & set(method_genes))
        overlap_percentage = (overlap / len(marker_gene_list)) * 100
        overlap_results_3rd_gran[method_name][cell_type] = overlap_percentage

In [ ]:
overlap_df_3rd_gran = pd.DataFrame(overlap_results_3rd_gran)
print("\nOverlap percentages with marker genes by cell type:")
overlap_df_3rd_gran

In [ ]:
plt.figure(figsize=(12, 6))

# Get positions for bars
x = np.arange(len(overlap_df_3rd_gran.index))
width = 0.2  # Width of bars

# Plot bars for each method
plt.bar(x - width*1.5, overlap_df_3rd_gran['Most variable genes'], width, label='Most variable genes')
plt.bar(x - width/2, overlap_df_3rd_gran['Low CV genes'], width, label='Low CV genes') 
plt.bar(x + width/2, overlap_df_3rd_gran['Highest R2 genes'], width, label='Highest R2 genes')
plt.bar(x + width*1.5, overlap_df_3rd_gran['Highest adjusted R2 genes'], width, label='Highest adjusted R2 genes')

# Customize plot
plt.xlabel('Cell Types')
plt.ylabel('Overlap with Marker Genes (%)')
plt.title('Comparison of Gene Selection Methods')
plt.xticks(x, overlap_df_3rd_gran.index, rotation=45, ha='right')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

## EXTRA: How much do the metrics dependent on the granularity vary?

In [ ]:
methods_2nd_gran = methods

# Calculate overlap between 2nd and 3rd granularity for highest R2 genes
r2_overlap = len(set(methods_2nd_gran['Highest R2 genes']) & set(methods_3rd_gran['Highest R2 genes']))
r2_overlap_percentage = (r2_overlap / 2000) * 100

# Calculate overlap between 2nd and 3rd granularity for highest adjusted R2 genes  
adj_r2_overlap = len(set(methods_2nd_gran['Highest adjusted R2 genes']) & set(methods_3rd_gran['Highest adjusted R2 genes']))
adj_r2_overlap_percentage = (adj_r2_overlap / 2000) * 100

print(f"Overlap between 2nd and 3rd granularity highest R2 genes: {r2_overlap_percentage:.2f}%")
print(f"Overlap between 2nd and 3rd granularity highest adjusted R2 genes: {adj_r2_overlap_percentage:.2f}%")
